#https://www.dha.com.tr/1560000 2659181


In [165]:
import aiohttp
import asyncio
import csv
import os
from bs4 import BeautifulSoup
import re
import time
from urllib.parse import urlparse

In [187]:

async def page_extractor(session, article_id):
    print(f"Processing article ID: {article_id}")
    url = f"https://www.dha.com.tr/{article_id}"
    try:
        async with session.get(url) as response:
            final_url = str(response.url)
            html = await response.text()
            soup = BeautifulSoup(html, 'html.parser')




        parsed_url = urlparse(final_url)
        path_parts = [part for part in parsed_url.path.split('/') if part]



        if path_parts[0] == str(article_id):
            path_parts[0] = "N/A"



        headline_tag = soup.find('h1')
        headline = headline_tag.get_text(strip=True) if headline_tag else "N/A"

        description_tag = soup.find('h2')
        description = description_tag.get_text(strip=True) if description_tag else "N/A"

        articleBody = "\n\n".join(
            p.get_text(strip=True)
            for p in soup.find_all('p')
            if not p.get_text(strip=True).startswith('Son Güncellenme')
        )

        if "Beklemek istemiyorsanıztıklayın" in articleBody:
            articleBody = "N/A"
            
        articleBody2=articleBody.replace("UYARI: www.dha.com.tr internet sitesinde yayınlanan yazı, haber ve fotoğrafların her türlü telif hakkı Demirören Ajansı A.Ş.’ye aittir. İzin alınmadan, kaynak gösterilerek dahi kullanılamaz.", "   ")



        meta_line_node = soup.find(string=re.compile(r'Son\s+Güncellenme'))
        meta_line = meta_line_node.strip() if meta_line_node else ''
        dates = re.findall(r'\d{2}\.\d{2}\.\d{4}', meta_line)
        dpublishdate = dates[0] if len(dates) >= 1 else "N/A"

        author_line = soup.find(string=re.compile(r'DHA\)'))
        if author_line:
            author_line = author_line.strip()
            author = author_line.split('|', 1)[1].split('/', 1)[0].strip() if '|' in author_line else "N/A"
        else:
            author = "N/A"

        if "DHA" in author:
            author = "N/A"


        return {
            "id": article_id,
            "headline": headline,
            "description": description,
            "articleBody": articleBody2,
            "dpublishdate": dpublishdate,
            "author": author,
            "articleType": path_parts[0],
        }
    except Exception as e:
        print(f"Error processing article {article_id}: {e}")
        return None





In [182]:

def get_processed_ids(csv_file):
    if not os.path.exists(csv_file):
        return set()
    with open(csv_file, encoding='utf-8') as f:
        reader = csv.DictReader(f)
        return {row['id'] for row in reader}

async def main(start, end):
    csv_file = 'dha_articles.csv'
    fieldnames = ["id", "headline", "description", "articleBody", "dpublishdate", "author", "articleType"]
    write_header = not os.path.exists(csv_file) or os.path.getsize(csv_file) == 0

    processed_ids = get_processed_ids(csv_file)
    ids_to_process = [i for i in range(start, end) if str(i) not in processed_ids]

    if not ids_to_process:
        print("No new articles to process.")
        return

    start_time = time.time()
    async with aiohttp.ClientSession() as session:
        tasks = [page_extractor(session, i) for i in ids_to_process]
        results = await asyncio.gather(*tasks)
        results = [r for r in results if r is not None]
    with open(csv_file, 'a', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        for row in results:
            writer.writerow(row)
    end_time = time.time()
    print(f"Time taken to get the details of profiles: {end_time - start_time} seconds")


In [188]:
async def main1(start,end):
    while(start <end):
        await main(start,start+100)
        start +=100


In [ ]:
await main1(1560000, 2659930)

Processing article ID: 1560027
Processing article ID: 1560072
Processing article ID: 1560079
Processing article ID: 1560083
Processing article ID: 1560088
Error processing article 1560027: 'utf-8' codec can't decode byte 0xc2 in position 36016: invalid continuation byte
Error processing article 1560088: 'utf-8' codec can't decode byte 0xc2 in position 36368: invalid continuation byte
Time taken to get the details of profiles: 7.131623983383179 seconds
Processing article ID: 1560103
Processing article ID: 1560114
Processing article ID: 1560132
Processing article ID: 1560163
Processing article ID: 1560164
Processing article ID: 1560173
Processing article ID: 1560186
Error processing article 1560173: 'utf-8' codec can't decode byte 0xc2 in position 36682: invalid continuation byte
Error processing article 1560164: 'utf-8' codec can't decode byte 0xc2 in position 36247: invalid continuation byte
Time taken to get the details of profiles: 0.8217272758483887 seconds
Processing article ID: 15